# Imports
---

In [ ]:
import numpy as np
import os
import subprocess
import numpy as np
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import seaborn as sns

from model import bbox_per_image, bbox_combined, bbox_combined_nms, binary_classifier, seg_accuracy
from utils import overlay

Dataset Info - Image Width: 16460, Image Height: 14590, Tile Size: 2048, Small Tile Size: 512


# Code Execution Sequence
---
1. Importing Required Libraries and Modules
2. Per Image based labels
3. Combined labels
4. Binary classification using spatial containment analysis
5. Overlaying the results on the orthophoto (Kamalapur)
6. Risk Level Analysis and Heatmap

# Object Detection and Building Segmentation
---
**Building Segmentation Postprocessings**

**Label Processing: Per Slice then Combined**

### YOLO

In [6]:
bbox_per_image.yolo_labels(mask_dir='resources/Building/masks/labels_11m_75', save_dir='resources/Building/bbox_perslice/YOLO-v11m-75p')

Processed All Labels: 4371


In [7]:
bbox_combined.process_sliding_window(label_dir='resources/Building/bbox_perslice/YOLO-v11m-75p', save_dir='resources/ClassifierOutput/Building Classification/YOLO-v11m-75p', SEG_MODEL='yolo')

x ------- Processing all bounding boxes commenced ------- x
Total bounding boxes before NMM: 4371
Total bounding boxes after NMM: 266
Total bounding boxes after containment post-processing: 252


**Using Greedy NMS**

In [ ]:
bbox_combined_nms.process_sliding_window(label_dir='resources/Building/bbox_perslice/YOLO-v11m', save_dir='resources/ClassifierOutput/Building Classification/YOLO-v11m-nms')

In [ ]:
binary_classifier.classify(building_labels_path='resources/ClassifierOutput/Building Classification/YOLO-v11m-nms/labels.txt', object_labels_path='resources/ClassifierOutput/Object Detection/YOLO-v9c/predicted_object_labels_v9gelanc.txt', output_labels_path='resources/ClassifierOutput/Building Classification/YOLO-v11m-nms/labels_binclass.txt')

### SegGPT

In [ ]:
bbox_per_image.seggpt_labels(mask_dir='resources/Building/masks/200_masks_seggpt', save_dir='resources/Building/bbox_perslice/SegGPT')

In [ ]:
bbox_combined.process_sliding_window(label_dir='resources/Building/bbox_perslice/SegGPT', save_dir='resources/ClassifierOutput/Building Classification/YOLO-v8m', SEG_MODEL='seggpt')

### Binary Classification (w/o distance-based inspection method)

In [ ]:
model = 'YOLO-v11m-25p'

binary_classifier.classify(building_labels_path=f'resources/ClassifierOutput/Building Classification/{model}/labels.txt', object_labels_path=f'resources/ClassifierOutput/Object Detection/YOLO-v9c/predicted_object_labels_v9gelanc.txt', output_labels_path=f'resources/ClassifierOutput/Building Classification/{model}/labels_binclass.txt')

61 / 216 objects are on top of buildings
Number of risky buildings: 39
Number of safe buildings: 205


**Binary Classification of Buildings**

In [ ]:
# Save in txt file: labels.txt
subprocess.run(["matlab", "-r", "model/seg_accuracy_nms.m; exit"])

# Risk Level Analysis & Assignment
---

In [ ]:
accuracy = {
    'construction_site': 0.0762,
    'flower_pot': 0.358,
    'open_tank': 0.303,
    'polythene': 0.309,
    'reservoir': 0.0,
    'tyres': 0.147
}

In [ ]:
from model import risk_level_analysis

risk_level_analysis.analyze_risks(building_labels_path='resources/ClassifierOutput/Building Classification/YOLO-v8m/building_classification.txt', object_labels_path='resources/ClassifierOutput/Object Detection/YOLO-v9c/predicted_object_labels_v9gelanc.txt', accuracy=accuracy, save_report_path='resources/ClassifierOutput/Building Classification/YOLO-v8m/report.csv')

In [ ]:
from utils import geo_coordinates_buildings

tiff_path = "resources/Orthophoto/Komlapur_Orthophoto.tif"
input_csv_path = "resources/ClassifierOutput/Building Classification/YOLO-v8m/report.csv"
output_csv_path = "resources/ClassifierOutput/Building Classification/YOLO-v8m/report_geocoor.csv"

# Run the function to generate geolocation and risk CSV
geo_coordinates_buildings.create_geolocation_risk_csv(tiff_path, input_csv_path, output_csv_path, accuracy)

In [ ]:
from utils import area_heatmap

# def create_risk_heatmap(building_data, image_width, image_height, output_path='riskmap.jpg', kernel_bandwidth=0.1):
area_heatmap.create_risk_heatmap('resources/ClassifierOutput/Building Classification/YOLO-v8m/report_geocoor.csv', image_width=16460, image_height=14590, output_path='resources/ClassifierOutput/Building Classification/YOLO-v8m/riskmap.jpg')

# Overlaying the Results on the Orthophoto
---

In [ ]:
overlay.draw(org_image_path='resources/Orthophoto/Komlapur_Orthophoto.jpg', building_labels_path='resources/ClassifierOutput/Building Classification/YOLO-v8m/labels_binclass_nms.txt', object_labels_path=None, output_path='output/overlay_nms.jpg')

In [ ]:
overlay.draw_segmentation(image_path='resources/Orthophoto/Komlapur_Orthophoto.jpg', label_path='resources/ClassifierOutput/Building Classification/YOLO-v8m-25p/labels.txt', output_path='output/overlay-v8m-25p.jpg')